# Notebook 3: Models

**What you'll learn:**
- What a "model" means in the SDK (it's an adapter/translator, not the AI itself)
- The Model abstract base class: `stream()` and `get_config()`
- How streaming works and what StreamEvents are
- How to write a custom callback handler
- How to switch between model providers
- How to configure BedrockModel

**Prerequisite:** Complete [NB2_The_Event_Loop.ipynb](./NB2_The_Event_Loop.ipynb)

**Companion reading:** `03-models.md`

---
## The Translator Analogy

Amazon Bedrock, OpenAI, and Anthropic all speak different API languages. The SDK needs a **common language**.

The `Model` class is a **translator**:

```
SDK                  Model Adapter              AI Provider
                     (Translator)
Messages -------->   BedrockModel ----------->  Amazon Bedrock API
 ToolSpecs ------->   (converts format)          (different format)
SystemPrompt ---->
                     <-----------
StreamEvents <----   (converts back)  <--------  Bedrock Response
```

Every model adapter does the same translation. That's why you can swap providers without changing your agent code.

---
## The Model Abstract Base Class

The `Model` class (`src/strands/models/model.py`) is an **Abstract Base Class (ABC)**. It defines the interface that all model adapters must implement.

Two key methods:
- **`stream()`** -- the main method. Sends messages to the AI and yields StreamEvents back.
- **`get_config()`** -- returns a dictionary of model settings.

`Model` itself cannot be used directly -- you must use a subclass like `BedrockModel`.

In [ ]:
# ============================================================
# Look at the Model ABC source code
# ============================================================

import inspect

# Import the base Model class
from strands.models.model import Model

# Print the stream() method signature.
# This is the abstract method that all model adapters must implement.
# It takes: messages, tool_specs, system_prompt, **kwargs
# It yields: StreamEvent objects
print("=== Model.stream() signature ===")
print(inspect.signature(Model.stream))
print()

# Print the get_config() method signature.
print("=== Model.get_config() signature ===")
print(inspect.signature(Model.get_config))
print()

# Show that Model is abstract -- you can't create it directly
print("=== Is Model abstract? ===")
print(f"Abstract methods: {Model.__abstractmethods__}")

In [ ]:
# ============================================================
# Create a BedrockModel with custom configuration
# ============================================================

from strands.models.bedrock import BedrockModel

# BedrockModel is a subclass of Model that translates to the
# Amazon Bedrock API format.
#
# Key parameters:
#   model_id:    which AI model to use on Bedrock
#   max_tokens:  maximum response length (in tokens)
#   region_name: AWS region

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514",  # Claude Sonnet on Bedrock
    max_tokens=1024,                                     # Limit response to 1024 tokens
)

# get_config() returns all settings as a dictionary.
# This is useful for logging or debugging.
print("=== BedrockModel Config ===")
config = model.get_config()
for key, value in config.items():
    print(f"  {key}: {value}")

---
## What are StreamEvents?

When the model generates a response, it doesn't send the entire text at once. It **streams** it in small chunks. Each chunk is a `StreamEvent`.

Types of StreamEvents:
- **Content chunks:** Pieces of the response text, arriving one at a time
- **Tool use:** The model wants to call a tool
- **Stop reason:** Why the model stopped (`end_turn`, `tool_use`, `max_tokens`)
- **Usage stats:** Token counts (input tokens, output tokens)

This is why you see text appear word-by-word when an agent responds. Each word (approximately) is a separate StreamEvent.

The **callback_handler** receives each StreamEvent and decides what to do with it. The default handler prints text to the console.

In [ ]:
# ============================================================
# Custom callback handler to see streaming in detail
# ============================================================

# A callback_handler is a callable (function or object with __call__)
# that receives streaming events from the model.
#
# We'll create a simple one that logs each event type.

# Counter to track how many chunks we receive
chunk_count = 0

def verbose_handler(**kwargs):
    """A callback handler that prints details about each streaming event."""
    global chunk_count
    
    # The handler receives keyword arguments.
    # Common keys: data, complete, current_tool_use, reasoningText
    
    if 'data' in kwargs:
        # 'data' contains a text chunk from the model's response.
        # This is the most common event -- one per word/token.
        chunk_count += 1
        text = kwargs['data']
        print(f"  [chunk {chunk_count:3d}] \"{text}\"", end="")
        if chunk_count % 5 == 0:
            print()  # newline every 5 chunks for readability

print("Custom handler defined. Let's use it.")

In [ ]:
# ============================================================
# Call an agent with our verbose handler
# ============================================================

from strands import Agent

# Reset counter
chunk_count = 0

# Create agent with our custom callback handler
agent_verbose = Agent(
    callback_handler=verbose_handler,
)

# Call the agent. You'll see each text chunk individually.
print("=== Streaming chunks ===")
result = agent_verbose("What is Python? Answer in exactly 2 sentences.")
print(f"\n\n=== Total chunks received: {chunk_count} ===")
print(f"\n=== Final result ===")
print(str(result))

---
## With vs Without Streaming

- **`callback_handler=<function>`** (default): Text appears in real-time, word by word
- **`callback_handler=None`**: Nothing appears until the agent is done. You get the result all at once.

Internally, the model ALWAYS streams. The callback handler just controls whether you see it in real-time.

In [ ]:
# ============================================================
# Same call without streaming
# ============================================================

# callback_handler=None means: don't show anything during generation.
# The model still streams internally, but the events are silently consumed.
agent_silent = Agent(callback_handler=None)

# Nothing will appear until the agent is completely done.
print("Calling agent (no streaming)...")
result_silent = agent_silent("What is Python? Answer in exactly 2 sentences.")
print("Done!")
print()
print(f"Result: {result_silent}")

---
## Available Model Providers

The SDK supports many AI providers. Each has its own adapter class:

| Provider | Class | What it is | Install |
|----------|-------|-----------|---------|
| Amazon Bedrock | `BedrockModel` | **Default.** AWS managed service. Claude, Nova, Llama, Mistral. | `pip install strands-agents` |
| Anthropic | `AnthropicModel` | Direct Anthropic API. Claude models. | `pip install strands-agents[anthropic]` |
| OpenAI | `OpenAIModel` | OpenAI API. GPT models. | `pip install strands-agents[openai]` |
| Google Gemini | `GeminiModel` | Google AI. Gemini models. | `pip install strands-agents[google]` |
| LiteLLM | `LiteLLMModel` | Universal adapter. 100+ providers. | `pip install strands-agents[litellm]` |
| Ollama | `OllamaModel` | Local models. Run on your machine. | `pip install strands-agents[ollama]` |
| SageMaker | `SageMakerModel` | AWS SageMaker endpoints. | `pip install strands-agents` |

**Same agent code, different models:**
```python
# These all work the same way:
agent = Agent(model=BedrockModel())
agent = Agent(model=AnthropicModel(model_id="claude-sonnet-4-20250514"))
agent = Agent(model=OpenAIModel(model_id="gpt-4o"))
agent = Agent(model=OllamaModel(model_id="llama3"))
```

In [ ]:
# ============================================================
# See what providers are available
# ============================================================

# BedrockModel is always available (included in base install)
from strands.models.bedrock import BedrockModel
print(f"BedrockModel: {BedrockModel.__name__} (always available)")

# Other providers require extra packages.
# We'll try to import each one and show its status.
providers = [
    ("strands.models.anthropic", "AnthropicModel", "strands-agents[anthropic]"),
    ("strands.models.openai", "OpenAIModel", "strands-agents[openai]"),
    ("strands.models.ollama", "OllamaModel", "strands-agents[ollama]"),
    ("strands.models.litellm", "LiteLLMModel", "strands-agents[litellm]"),
]

for module_name, class_name, install_cmd in providers:
    try:
        module = __import__(module_name, fromlist=[class_name])
        cls = getattr(module, class_name)
        print(f"{class_name}: available")
    except ImportError:
        print(f"{class_name}: not installed (pip install {install_cmd})")

---
## Summary

What you learned in this notebook:

- **Model = translator** between SDK's language and the provider's API
- **Model ABC** defines `stream()` (abstract) and `get_config()` -- all providers implement this
- **StreamEvents** are chunks of the response arriving one at a time (text, tool_use, stop_reason, usage)
- **callback_handler** receives StreamEvents in real-time; set to `None` to disable
- **Multiple providers** available: Bedrock (default), Anthropic, OpenAI, Gemini, LiteLLM, Ollama, SageMaker
- **Swapping models** is easy: just change the `model=` parameter, everything else stays the same

**Key source files:**
| File | What it does |
|------|--------------|
| `src/strands/models/model.py` | Model ABC |
| `src/strands/models/bedrock.py` | Amazon Bedrock adapter |
| `src/strands/types/streaming.py` | StreamEvent types |

**Next:** [NB4_Tools.ipynb](./NB4_Tools.ipynb) -- The tool system: @tool decorator, ToolSpec, ToolRegistry, ToolContext